In [1]:
# !pip install jsonschema
import json
from jsonschema import validate
from constants import ValidationError, ErrorType, assert_all_logs
from engine import Engine
import os

data_file = 'data.json'
schema_file = 'schema.json'
data_enum_error_file = 'data_enum_error.json'
data_type_error_file = 'data_type_error.json'
data_no_schema_error_file = 'data_no_schema_error.json'
data_unclosed_error_file = 'data_unclosed_error.json'

In [2]:
json_data = json.load(open(data_file))
json_schema = json.load(open(schema_file))

# Validate the JSON data against the schema using jsonschema library
# jsonschema requires valid python dict for both data and schema
validate(instance=json_data, schema=json_schema)

In [3]:
def verify_json(schema=schema_file, data=data_file, max_depth = None):
    engine = Engine(schema=schema, target=data, max_depth=max_depth)
    log = engine.run()
    return log

In [4]:
# valid json data verification
verify_json(data=data_file)

[]

In [5]:
# type error verification
log = verify_json(data=data_type_error_file)
target_errors = {
    'top_object.a': ValidationError(ErrorType.BAD_VALUE, {'value': 'c', 'rule': 'type(number)'}),
    'top_object': ValidationError(ErrorType.INCOMPLETE, {'value': 'a'})
}

# print(log)
# assert_all_logs(log, target_errors)
log

pop: top_object.a ->     <BAD VALUE>: value(c) violates schema[type(number)]
(<class 'constants.ValidationResult'>)
start to add log: top_object.a c
add log to logs:     <BAD VALUE>: value(c) violates schema[type(number)]

pop: top_object ->     <INCOMPLETE>: child(a) is not valid
(<class 'constants.ValidationResult'>)
start to add log: top_object failed children: a
add log to logs:     <INCOMPLETE>: child(a) is not valid



[invalid json item: path(top_object.a)
   node info: c
     <BAD VALUE>: value(c) violates schema[type(number)],
 invalid json item: path(top_object)
   node info: failed children: a
     <INCOMPLETE>: child(a) is not valid]

In [6]:
# unclosed error verification
log = verify_json(data=data_unclosed_error_file)

target_errors = {
    'top_object.d': ValidationError(ErrorType.UNCLOSED),
    'top_object': ValidationError(ErrorType.UNCLOSED)
}

# assert_all_logs(log, target_errors)
log

add log to logs:     <UNCLOSED>: unclosed structure

add log to logs:     <UNCLOSED>: unclosed structure



[invalid json item: path(top_object.d)
     <UNCLOSED>: unclosed structure,
 invalid json item: path(top_object)
     <UNCLOSED>: unclosed structure]

In [7]:
# maximum stack depth verification
log = verify_json(data=data_file, max_depth=2)

target_errors = {
    'circuit_breaker': ValidationError(ErrorType.DEPTH_ERROR, {'depth': 2})
}
# assert_all_logs(log, target_errors)
log

add log to logs: <DEPTH ERROR>: maximum allowed depth exceeded: 2


[invalid json item: path(circuit_breaker)
     <DEPTH ERROR>: maximum allowed depth exceeded: 2]

In [10]:
# enumeration error verification
log = verify_json(data=data_enum_error_file)
target_errors = {
    'top_object.d[2]': ValidationError(ErrorType.BAD_VALUE, {'value': 5, 'rule': 'enum([3, 4])'}),
    'top_object.d': ValidationError(ErrorType.INCOMPLETE, {'value': 2}),
    'top_object': ValidationError(ErrorType.INCOMPLETE, {'value': 'd'})
}

# assert_all_logs(log, target_errors)
log

pop: top_object.d[2] ->     <BAD VALUE>: value(5) violates schema[enum([3, 4])]
(<class 'constants.ValidationResult'>)
start to add log: top_object.d[2] 5
add log to logs:     <BAD VALUE>: value(5) violates schema[enum([3, 4])]

pop: top_object.d ->     <INCOMPLETE>: child(2) is not valid
(<class 'constants.ValidationResult'>)


[invalid json item: path(top_object.d[2])
   node info: 5
     <BAD VALUE>: value(5) violates schema[enum([3, 4])]]

In [9]:
# unexpected json object verification
log = verify_json(data=data_no_schema_error_file)

target_errors = {
    'top_object.b.e': ValidationError(ErrorType.UNEXPECTED, {'value': 'extra'}),
    'top_object.b': ValidationError(ErrorType.INCOMPLETE, {'value': 'e'}),
    'top_object': ValidationError(ErrorType.INCOMPLETE, {'value': 'b'})
}

# assert_all_logs(log, target_errors)
log

pop: top_object.b.e -> <UNEXPECTED>: unexpected: extra(<class 'constants.ValidationError'>)
start to add log: top_object.b.e extra
add log to logs: <UNEXPECTED>: unexpected: extra
pop: top_object.b ->     <INCOMPLETE>: child(e) is not valid
(<class 'constants.ValidationResult'>)
start to add log: top_object.b failed children: e
add log to logs:     <INCOMPLETE>: child(e) is not valid

pop: top_object ->     <INCOMPLETE>: child(b) is not valid
(<class 'constants.ValidationResult'>)
start to add log: top_object failed children: b
add log to logs:     <INCOMPLETE>: child(b) is not valid



[invalid json item: path(top_object.b.e)
   node info: extra
     <UNEXPECTED>: unexpected: extra,
 invalid json item: path(top_object.b)
   node info: failed children: e
     <INCOMPLETE>: child(e) is not valid,
 invalid json item: path(top_object)
   node info: failed children: b
     <INCOMPLETE>: child(b) is not valid]